In [ ]:
import pandas as pd


df = pd.read_csv('/content/student_performance.csv')


display(df.head())

In [ ]:
import sqlite3


conn = sqlite3.connect('student_analytics.db')


df.to_sql('students', conn, if_exists='replace', index=False)

print("DataFrame successfully stored in 'student_analytics.db' as table 'students'.")


conn.close()

In [ ]:
import sqlite3
import pandas as pd


conn = sqlite3.connect('student_analytics.db')


query_1 = "SELECT * FROM students;"
df_query_1 = pd.read_sql_query(query_1, conn)

print("Query 1: All student data (first 5 rows):")
display(df_query_1.head())


conn.close()

In [ ]:
import sqlite3
import pandas as pd


conn = sqlite3.connect('student_analytics.db')


query_2 = """
SELECT
    department,
    AVG(math_score) AS avg_math_score,
    AVG(science_score) AS avg_science_score,
    AVG(english_score) AS avg_english_score,
    AVG(programming_score) AS avg_programming_score
FROM students
GROUP BY department;
"""
df_query_2 = pd.read_sql_query(query_2, conn)
print("\nQuery 2: Average scores per department:")
display(df_query_2)


query_3 = """
SELECT
    gender,
    COUNT(student_id) AS num_students,
    AVG(attendance_percentage) AS avg_attendance_percentage
FROM students
GROUP BY gender;
"""
df_query_3 = pd.read_sql_query(query_3, conn)
print("\nQuery 3: Student count and average attendance by gender:")
display(df_query_3)


query_4 = """
SELECT
    name,
    (math_score + science_score + english_score + programming_score) AS total_score
FROM students
ORDER BY total_score DESC
LIMIT 5;
"""
df_query_4 = pd.read_sql_query(query_4, conn)
print("\nQuery 4: Top 5 students by total score:")
display(df_query_4)


query_5 = """
SELECT
    city,
    COUNT(student_id) AS num_students
FROM students
GROUP BY city
ORDER BY num_students DESC;
"""
df_query_5 = pd.read_sql_query(query_5, conn)
print("\nQuery 5: Student distribution by city:")
display(df_query_5)


conn.close()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


sns.set_style("whitegrid")

plt.figure(figsize=(12, 6))
df_query_2.set_index('department').plot(kind='bar', figsize=(12, 7))
plt.title('Average Scores per Department')
plt.ylabel('Average Score')
plt.xlabel('Department')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
sns.barplot(x='gender', y='avg_attendance_percentage', data=df_query_3, palette='viridis')
plt.title('Average Attendance Percentage by Gender')
plt.ylabel('Average Attendance Percentage')
plt.xlabel('Gender')
plt.ylim(0, 100)
plt.tight_layout()
plt.show()


plt.figure(figsize=(8, 6))
sns.barplot(x='total_score', y='name', data=df_query_4, palette='magma')
plt.title('Top 5 Students by Total Score')
plt.xlabel('Total Score')
plt.ylabel('Student Name')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import numpy as np


ml_df = df.drop(['student_id', 'name'], axis=1)


label_encoders = {}
for column in ['gender', 'department', 'city']:
    le = LabelEncoder()
    ml_df[column] = le.fit_transform(ml_df[column])
    label_encoders[column] = le

X = ml_df.drop('programming_score', axis=1)
y = ml_df['programming_score']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nRandom Forest Model Performance:")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R2): {r2:.2f}")

feature_importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nFeature Importances:")
display(feature_importances)